# Load Datasets 


In [1]:
import pandas as pd
import numpy as np

bat = pd.read_csv(r"C:\Users\shiv_\OneDrive\Desktop\Cricket Analytics & Player Performance Intelligence Dashboard\Data\Processed\Test_Batting_Cleaned.csv")
bowl = pd.read_csv(r"C:\Users\shiv_\OneDrive\Desktop\Cricket Analytics & Player Performance Intelligence Dashboard\Data\Processed\Bowling_test_Cleaned.csv")
field = pd.read_csv(r"C:\Users\shiv_\OneDrive\Desktop\Cricket Analytics & Player Performance Intelligence Dashboard\Data\Processed\Fielding_test_Cleaned.csv")

bat = bat.dropna(subset=['Player'])
bowl = bowl.dropna(subset=['Player'])
field = field.dropna(subset=['Player'])
print("Shape of Batting data:", bat.shape)
print("Shape of Bowling data:", bowl.shape)
print("Shape of Fielding data:", field.shape)

Shape of Batting data: (3001, 19)
Shape of Bowling data: (3050, 22)
Shape of Fielding data: (3001, 17)


In [2]:
bat_players = set(bat['Player'])
bowl_players = set(bowl['Player'])
field_players = set(field['Player'])

print("Unique in batting:", len(bat_players))
print("Unique in bowling:", len(bowl_players))
print("Unique in fielding:", len(field_players))

print("Batting ∩ Bowling:", len(bat_players & bowl_players))
print("Batting ∩ Fielding:", len(bat_players & field_players))
print("Bowling ∩ Fielding:", len(bowl_players & field_players))

Unique in batting: 2995
Unique in bowling: 50
Unique in fielding: 2995
Batting ∩ Bowling: 50
Batting ∩ Fielding: 2995
Bowling ∩ Fielding: 50


In [3]:
print(bowl['Player'].value_counts().head(20))

Player
M Muralitharan (ICC/SL)    61
SK Warne (AUS)             61
A Kumble (INDIA)           61
JM Anderson (ENG)          61
GD McGrath (AUS)           61
CA Walsh (WI)              61
SCJ Broad (ENG)            61
DW Steyn (SA)              61
N Kapil Dev (INDIA)        61
HMRKB Herath (SL)          61
Sir RJ Hadlee (NZ)         61
SM Pollock (SA)            61
Harbhajan Singh (INDIA)    61
Wasim Akram (PAK)          61
CEL Ambrose (WI)           61
M Ntini (SA)               61
IT Botham (ENG)            61
NM Lyon (AUS)              61
MD Marshall (WI)           61
Waqar Younis (PAK)         61
Name: count, dtype: int64


In [7]:
sample_name = bowl['Player'].value_counts().index[0]
rows = bowl[bowl['Player'] == sample_name]
print(rows.shape)
print(rows.drop_duplicates().shape)

(61, 22)
(1, 22)


In [8]:
bowl = bowl.drop_duplicates(subset='Player').reset_index(drop=True)
print(bowl.shape)

(50, 22)


In [9]:
print(bat.duplicated(subset='Player').sum())
dup_names_bat = bat[bat.duplicated(subset='Player', keep=False)]['Player'].unique()
print(dup_names_bat)

print(field.duplicated(subset='Player').sum())
dup_names_field = field[field.duplicated(subset='Player', keep=False)]['Player'].unique()
print(dup_names_field)

6
['Imran Khan (PAK)' 'P Roy (INDIA)' 'JP Duminy (SA)' 'A Ward (ENG)'
 'D Pretorius (SA)' 'RA Austin (WI)']
6
['JP Duminy (SA)' 'Imran Khan (PAK)' 'P Roy (INDIA)' 'RA Austin (WI)'
 'A Ward (ENG)' 'D Pretorius (SA)']


In [10]:
example = dup_names_bat[0]
print(bat[bat['Player'] == example])

                Player       Span  Start_Year  End_Year  Career_Length  Mat  \
142   Imran Khan (PAK)  1971-1992        1971      1992             21   88   
2534  Imran Khan (PAK)  2014-2019        2014      2019              5   10   

     Inns  NO  Runs   HS HS_Numeric  HS_NotOut    Ave 100  50  0  \
142   126  25  3807  136        136          0  37.69   6  18  8   
2534   10   3    16    6          6          0   2.28   0   0  5   

     Runs_Per_Match Century_Rate Fifty_Rate  
142           43.26         6.82      20.45  
2534           1.60         0.00       0.00  


In [11]:
for name in dup_names_bat:
    print(bat[bat['Player'] == name][['Player', 'Span', 'Start_Year', 'End_Year', 'Mat']])
    print()

                Player       Span  Start_Year  End_Year  Mat
142   Imran Khan (PAK)  1971-1992        1971      1992   88
2534  Imran Khan (PAK)  2014-2019        2014      2019   10

             Player       Span  Start_Year  End_Year  Mat
257   P Roy (INDIA)  1951-1960        1951      1960   43
1925  P Roy (INDIA)  1982-1982        1982      1982    2

              Player       Span  Start_Year  End_Year  Mat
293   JP Duminy (SA)  2008-2017        2008      2017   46
2306  JP Duminy (SA)  1927-1929        1927      1929    3

            Player       Span  Start_Year  End_Year  Mat
883   A Ward (ENG)  1893-1895        1893      1895    7
2202  A Ward (ENG)  1969-1976        1969      1976    5

                Player       Span  Start_Year  End_Year  Mat
2199  D Pretorius (SA)  2019-2019        2019      2019    1
2433  D Pretorius (SA)  2002-2003        2002      2003    4

              Player       Span  Start_Year  End_Year  Mat
2203  RA Austin (WI)  2009-2009        2009     

In [12]:
bat['player_key'] = bat['Player'] + '_' + bat['Start_Year'].astype(str)
bowl['player_key'] = bowl['Player'] + '_' + bowl['Start_Year'].astype(str)
field['player_key'] = field['Player'] + '_' + field['Start_Year'].astype(str)

print(bat.duplicated(subset='player_key').sum())
print(bowl.duplicated(subset='player_key').sum())
print(field.duplicated(subset='player_key').sum())

0
0
0


In [13]:
bat_keys = set(bat['player_key'])
bowl_keys = set(bowl['player_key'])
field_keys = set(field['player_key'])

print("Batting ∩ Bowling:", len(bat_keys & bowl_keys))
print("Batting ∩ Fielding:", len(bat_keys & field_keys))

Batting ∩ Bowling: 50
Batting ∩ Fielding: 3001


# Merge 

In [14]:
merged_test = bat.merge(bowl, on='player_key', how='outer', suffixes=('_bat', '_bowl'))
print(merged_test.shape)
print(merged_test.shape[0], "vs union:", len(bat_keys | bowl_keys))

(3001, 42)
3001 vs union: 3001


In [15]:
merged_test = merged_test.merge(field, on='player_key', how='outer', suffixes=('', '_field'))
print(merged_test.shape)
print(merged_test.shape[0], "vs union:", len(bat_keys | bowl_keys | field_keys))

(3001, 59)
3001 vs union: 3001


In [16]:
merged_test['Player'] = merged_test['Player_bat'].fillna(merged_test['Player_bowl']).fillna(merged_test['Player'])
merged_test['Span'] = merged_test['Span_bat'].fillna(merged_test['Span_bowl']).fillna(merged_test['Span'])
merged_test['Start_Year'] = merged_test['Start_Year_bat'].fillna(merged_test['Start_Year_bowl']).fillna(merged_test['Start_Year'])
merged_test['End_Year'] = merged_test['End_Year_bat'].fillna(merged_test['End_Year_bowl']).fillna(merged_test['End_Year'])

In [17]:
cols_to_drop = ['Player_bat', 'Player_bowl', 'Span_bat', 'Span_bowl', 'Start_Year_bat', 'Start_Year_bowl', 'End_Year_bat', 'End_Year_bowl']
merged_test = merged_test.drop(columns=[c for c in cols_to_drop if c in merged_test.columns])
print(merged_test.shape)

(3001, 51)


In [18]:
merged_test['has_batting'] = merged_test['Mat_bat'].notna()
merged_test['has_bowling'] = merged_test['Mat_bowl'].notna()

merged_test['player_role_hint'] = 'Unknown'
merged_test.loc[merged_test['has_batting'] & ~merged_test['has_bowling'], 'player_role_hint'] = 'Batsman'
merged_test.loc[~merged_test['has_batting'] & merged_test['has_bowling'], 'player_role_hint'] = 'Bowler'
merged_test.loc[merged_test['has_batting'] & merged_test['has_bowling'], 'player_role_hint'] = 'All-rounder'

print(merged_test['player_role_hint'].value_counts())

player_role_hint
Batsman        2951
All-rounder      50
Name: count, dtype: int64


In [19]:
merged_test.info()
merged_test.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3001 entries, 0 to 3000
Data columns (total 54 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Career_Length_bat     3001 non-null   int64  
 1   Mat_bat               3001 non-null   int64  
 2   Inns_bat              3001 non-null   object 
 3   NO                    3001 non-null   object 
 4   Runs_bat              3001 non-null   object 
 5   HS                    3001 non-null   object 
 6   HS_Numeric            3001 non-null   object 
 7   HS_NotOut             3001 non-null   int64  
 8   Ave_bat               3001 non-null   object 
 9   100                   3001 non-null   object 
 10  50                    3001 non-null   object 
 11  0                     3001 non-null   object 
 12  Runs_Per_Match        3001 non-null   object 
 13  Century_Rate          3001 non-null   object 
 14  Fifty_Rate            3001 non-null   object 
 15  player_key           

,Career_Length_bat,Mat_bat,Inns_bat,NO,Runs_bat,HS,HS_Numeric,HS_NotOut,Ave_bat,100,...,Ct Wk,Ct Fi,MD,D/I,Catches_Per_Match,Dismissals_Per_Match,Stumping_Rate,has_batting,has_bowling,player_role_hint
115,2,10,12,4,410,65,65,0,51.25,0,...,0,15,3 (3ct 0st),0.789,1.50,1.50,0.0,True,False,Batsman
2887,15,13,15,8,37,18*,18,1,5.28,0,...,0,2,1 (1ct 0st),0.086,0.15,0.15,0.0,True,False,Batsman
2660,0,1,1,0,1,1,1,0,1,0,...,0,1,1 (1ct 0st),0.500,1.00,1.00,0.0,True,False,Batsman
1141,8,35,68,6,1404,155*,155,1,22.64,1,...,0,20,2 (2ct 0st),0.307,0.57,0.57,0.0,True,False,Batsman
1830,2,4,6,1,37,15,15,0,7.4,0,...,0,1,1 (1ct 0st),0.166,0.25,0.25,0.0,True,False,Batsman
2138,3,9,11,2,47,10*,10,1,5.22,0,...,0,4,1 (1ct 0st),0.222,0.44,0.44,0.0,True,False,Batsman
2874,8,23,37,3,879,116,116,0,25.85,2,...,0,8,1 (1ct 0st),0.181,0.35,0.35,0.0,True,False,Batsman
2079,1,4,6,2,70,26,26,0,17.5,0,...,0,1,1 (1ct 0st),0.142,0.25,0.25,0.0,True,False,Batsman
2307,1,4,7,0,143,30,30,0,20.42,0,...,0,1,1 (1ct 0st),0.125,0.25,0.25,0.0,True,False,Batsman
1009,5,9,14,3,158,70*,70,1,14.36,0,...,0,3,1 (1ct 0st),0.200,0.33,0.33,0.0,True,False,Batsman


In [20]:
merged_test.to_csv(r"../Data/Processed/Test_Player_Profile.csv", index=False)